# Assignment 2: Experiment Tracking with MLflow

This notebook covers the model training pipeline, hyperparameter tuning, and experiment tracking using MLflow.  
We rigorously test models across different data versions to ensure robustness.

In [9]:
!pip install protobuf --upgrade 
!pip install mlflow scikit-learn pandas numpy 

In [10]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import average_precision_score, precision_recall_curve, auc
import os

# Initialize MLflow Experiment
mlflow.set_experiment("SMS_Spam_Detection_v2")
mlflow.set_tracking_uri("file:./mlruns")

2026/02/16 00:15:52 INFO mlflow.tracking.fluent: Experiment with name 'SMS_Spam_Detection_v2' does not exist. Creating a new experiment.


In [11]:
# Helper: Load Data
def load_data():
    if not os.path.exists('train.csv'):
        raise FileNotFoundError("Data files not found. Ensure prepare.ipynb has been run.")
    
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
    
    X_train = train_df['message']
    y_train = train_df['label']
    X_test = test_df['message']
    y_test = test_df['label']
    
    return X_train, y_train, X_test, y_test

In [6]:
# Define Pipelines & Parameter Grids

def get_model_configs():
    # 1. Logistic Regression
    pipe_lr = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ])
    param_lr = {
        'clf__C': [0.1, 1.0, 5.0, 10.0],
        'clf__solver': ['liblinear', 'lbfgs']
    }

    # 2. Random Forest
    pipe_rf = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('clf', RandomForestClassifier(random_state=42))
    ])
    param_rf = {
        'clf__n_estimators': [50, 100],
        'clf__max_depth': [10, 20, None]
    }

    # 3. SVM (Support Vector Machine)
    pipe_svc = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('clf', SVC(probability=True, random_state=42))
    ])
    param_svc = {
        'clf__C': [0.1, 1.0, 2.0],
        'clf__kernel': ['linear', 'rbf']
    }

    return [
        ("LogisticRegression", pipe_lr, param_lr),
        ("RandomForest", pipe_rf, param_rf),
        ("SVM", pipe_svc, param_svc)
    ]

In [12]:
def run_training_cycle(data_version_name):
    print(f"\n=== Starting Training Cycle: {data_version_name} ===")
    X_train, y_train, X_test, y_test = load_data()
    
    # Parent run for this data version
    with mlflow.start_run(run_name=f"Cycle_{data_version_name}"):
        mlflow.log_param("data_version", data_version_name)
        
        configs = get_model_configs()
        
        for model_name, pipeline, param_grid in configs:
            with mlflow.start_run(run_name=f"{model_name}_{data_version_name}", nested=True):
                print(f"Training {model_name}...")
                
                # GridSearchCV for tuning
                grid = GridSearchCV(pipeline, param_grid, cv=3, scoring='average_precision', n_jobs=-1)
                grid.fit(X_train, y_train)
                
                best_model = grid.best_estimator_
                
                # Evaluation on Test Set
                # Predict probabilities for AUCPR
                if hasattr(best_model, "predict_proba"):
                    probs = best_model.predict_proba(X_test)[:, 1]
                else:
                    probs = best_model.decision_function(X_test)
                
                precision, recall, _ = precision_recall_curve(y_test, probs)
                aucpr_score = auc(recall, precision)
                
                # Log Parameters & Metrics
                mlflow.log_params(grid.best_params_)
                mlflow.log_metric("test_aucpr", aucpr_score)
                
                # Log Model
                mlflow.sklearn.log_model(best_model, "model")
                
                # Print results
                print(f"  -> {model_name} Test AUCPR: {aucpr_score:.4f}")

In [13]:
# Execute Full Workflow

# 1. Train on Version 1
!git checkout v1
!dvc checkout
run_training_cycle("v1_Seed42")

# 2. Train on Version 2
!git checkout v2
!dvc checkout
run_training_cycle("v2_Seed100")

M	Assignment 2/prepare.ipynb
Previous HEAD position was 142125a Data V2: Seed 100
HEAD is now at 16b14a5 Data V1: Seed 42
zsh:1: command not found: dvc

=== Starting Training Cycle: v1_Seed42 ===
Training LogisticRegression...


2026/02/16 00:16:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/mohit/anaconda3/envs/envstreamlit/lib/python3.12/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


  -> LogisticRegression Test AUCPR: 0.9739
Training RandomForest...


2026/02/16 00:16:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/mohit/anaconda3/envs/envstreamlit/lib/python3.12/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


  -> RandomForest Test AUCPR: 0.9787
Training SVM...


...

  -> LogisticRegression Test AUCPR: 0.9739
Training RandomForest...


2026/02/16 00:16:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/mohit/anaconda3/envs/envstreamlit/lib/python3.12/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


  -> RandomForest Test AUCPR: 0.9787
Training SVM...


2026/02/16 00:17:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/mohit/anaconda3/envs/envstreamlit/lib/python3.12/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


  -> SVM Test AUCPR: 0.9807


In [14]:
# Champion Model Selection
print("\n=== Champion Model Selection ===")

# Query MLflow for the best run
runs = mlflow.search_runs(order_by=["metrics.test_aucpr DESC"])
best_run = runs.iloc[0]

print(f"Champion Model ID: {best_run.run_id}")
print(f"Best AUCPR: {best_run['metrics.test_aucpr']:.4f}")
print(f"Parameters: {best_run.filter(like='params').to_dict()}")


=== Champion Model Selection ===
Champion Model ID: 2a4e2fb9916c45ca813afb4eafffa325
Best AUCPR: 0.9807
Parameters: {'params.clf__kernel': 'rbf', 'params.clf__C': '2.0', 'params.clf__n_estimators': None, 'params.clf__max_depth': None, 'params.clf__solver': None, 'params.data_version': None}
